# 3. 로지스틱 회귀 타이타닉 생존 예측 (Logistic Regression Titanic)

타이타닉 승객 데이터를 전처리하고 `LogisticRegression` 학습, 성능 평가 및 확률 임계값 조정 효과를 확인합니다.

In [ ]:
from pathlib import Path
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

data_dir = DATA_DIR if 'DATA_DIR' in globals() else Path('data')
if not (data_dir / 'titanic.csv').is_file():
    data_dir = Path('Study_Python') / '[8] Model_Evaluation' / 'data'
if not (data_dir / 'titanic.csv').is_file():
    data_dir = Path('..') / 'Study_Python' / '[8] Model_Evaluation' / 'data'

titanic_path = data_dir / 'titanic.csv'
data = pd.read_csv(titanic_path) if titanic_path.is_file() else sns.load_dataset('titanic')

## 문제 1. 전처리, 누수 방지 결측 대체·스케일링 및 로지스틱 회귀 학습

`sex`를 `LabelEncoder`로 인코딩한 뒤 `['pclass', 'sex', 'age', 'fare']` 피처로 `survived`를 예측하세요. (`test_size=0.2, random_state=0`으로 먼저 분할한 뒤, `X_train`의 중앙값으로 `X_train`/`X_test` 결측을 대체하고 `StandardScaler`도 훈련 데이터에만 `fit`)

In [ ]:
df = data.copy()
df['sex'] = LabelEncoder().fit_transform(df['sex'].astype(str))
X = df[['pclass', 'sex', 'age', 'fare']]
y = df['survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
train_medians = X_train.median(numeric_only=True)
X_train = X_train.fillna(train_medians)
X_test = X_test.fillna(train_medians)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)
model = LogisticRegression()
model.fit(X_train_sc, y_train)
y_pred = model.predict(X_test_sc)
print('Accuracy:', round(accuracy_score(y_test, y_pred), 4))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, digits=4))

**결과 해석**

분할 후 훈련 세트(`X_train`)의 중앙값과 평균·표준편차만 사용해 결측 대체 및 스케일링을 적용함으로써 테스트 세트 정보가 전처리에 섞이는 데이터 누수를 방지하면서 약 80% 수준의 분류 정확도를 얻습니다.

## 문제 2. 예측 확률 임계값(threshold=0.7) 조정 비교

학습된 모델의 `predict_proba(X_test_sc)[:, 1]`을 구해 기본 임계값(0.5)과 상향 임계값(0.7)일 때의 `confusion_matrix`와 `classification_report`를 비교하세요.

In [ ]:
df = data.copy()
df['sex'] = LabelEncoder().fit_transform(df['sex'].astype(str))
X = df[['pclass', 'sex', 'age', 'fare']]
y = df['survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
train_medians = X_train.median(numeric_only=True)
X_train = X_train.fillna(train_medians)
X_test = X_test.fillna(train_medians)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)
model = LogisticRegression().fit(X_train_sc, y_train)

y_prob = model.predict_proba(X_test_sc)[:, 1]
y_pred_07 = (y_prob >= 0.7).astype(int)
print('Threshold 0.7 Confusion Matrix:\n', confusion_matrix(y_test, y_pred_07))
print(classification_report(y_test, y_pred_07, digits=4))

**결과 해석**

임계값을 0.5에서 0.7로 높이면 생존(1)으로 예측하는 기준이 엄격해져 양성 정밀도(Precision)는 높아지지만 실제 생존자를 놓칠 수 있어 재현율(Recall)은 낮아집니다.